# MyoChallenge 2026 — ChaseTag quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.6_ChaseTag_MyoChallenge.ipynb)

For **first-time invitees**. About 10 minutes:

1. Register on the **website** (not in this notebook)
2. Copy your **CLI token** from Submissions
3. Export the [tutorial 5.4](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.4_MuscleMimic_Directional_Locomotion.ipynb) walker to ONNX and submit it
4. Watch the result on the **website**
5. Train a stronger player in 5.4, then re-export
6. Optional: **play** `myoChallengeChaseTagFBVs-v0` locally (score + render)

Site: [mc26.myosuite.org](https://mc26.myosuite.org)

Invitation only. If the site says your email is not on the list, [apply here](https://docs.google.com/forms/d/e/1FAIpQLSf0tsCktz3adRUB6YGd0OfQMP9lQDrzeHeZCozrlVDN2J6_qw/viewform) first.


## What you submit

Each team submits **one file: a single player**.

You do **not** submit an opponent. The server always provides the other player.

| You send | The server does |
|---|---|
| **One** ONNX model | Runs it as player A |
| | Puts a champion (or another team) as player B |
| | Scores the match |

The file is a neural net that, every step:

- **reads** 546 numbers — your body, the opponent’s **hip** (not their muscles), and a **chase/evade flag**
- **writes** 354 numbers — your muscle activations, each between 0 and 1

```
obs (546)  →  YOUR MODEL  →  action (354)
```

You do **not** pick chase vs evade. One file plays both. The server assigns the role each episode and writes it into the observation. Opponent hip is only **where they are** — not whether to hunt or flee.

| Slice | What it is |
|---|---|
| `obs[0:537]` | Your body (joints, velocities, muscles, pelvis) |
| `obs[537:544]` | Opponent hip relative pose (where they are) |
| `obs[544:546]` | **Role** — `[1, 0]` = **chase** (go toward them), `[0, 1]` = **evade** (run away) |

Do not infer chase/evade from hip position. The same offset means opposite actions depending on those last two numbers. The export cell already reads `obs[544:546]` and flips heading.

That same file is ranked against a fixed champion during the cycle, then used in a 1v1 tournament against other teams.


## 1. Register on the website

Do this in the browser, not in Colab.

1. Open [mc26.myosuite.org](https://mc26.myosuite.org).
2. Click **Register your team**.
3. Fill **team name**, **email**, and **password** (8+ characters). Country and education are optional.
4. Click **Register team**. Already registered? Use **Sign in**.

<p><img alt="Register and Sign in on mc26.myosuite.org" width="480" src="https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/colab-auth.png"></p>


## 2. Copy your CLI token

On the website, open **Submissions** and copy the **CLI submission token**. Paste it with your email into the Colab cell below. Do not share the token.

<p><img alt="CLI submission token on Submissions" width="440" src="https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/colab-token.png"></p>


## 3. Export the 5.4 walker and submit it

The cells below take the tutorial 5.4 walker (`policy_bc_best.pt`), wrap it for ChaseTag, and upload the ONNX.

The shipped `policy_heading_table.pt` (5.4 walker + per-heading left-leg bias, made for musclemimic_models 1.0.6) exports the same way.

The walker reads **528** numbers. The server sends **546**. `export_fbvs_onnx` (`myosuite.integrations.musclemimic.fbvs_player`) maps 546 → 528: heading from opponent hip XY, **flipped by the role flag** `obs[544:546]` (`[1, 0]` chase toward them, `[0, 1]` evade away). Do not export the raw 528-d net.

If you already trained 5.4, upload your own `policy_bc_best.pt` or `policy_heading_table.pt` into this Colab first — that file is used instead of the download.

Paste email + token, then **Runtime → Run all**.

In [ ]:
# @title Install
%pip -q install -U "evalmc>=0.0.4" onnx "myosuite @ git+https://github.com/MyoHub/myosuite.git@dev"
print("ok")

In [ ]:
# @title Paste email + CLI token from the website
EMAIL = ""  # @param {type:"string"}
AUTH_TOKEN = ""  # @param {type:"string"}

EMAIL = EMAIL.strip().lower()
AUTH_TOKEN = AUTH_TOKEN.strip()
if not EMAIL or not AUTH_TOKEN:
    raise SystemExit("Paste EMAIL and AUTH_TOKEN from mc26.myosuite.org → Submissions.")
print("email", EMAIL, "token length", len(AUTH_TOKEN))


In [ ]:
# @title Export tutorial 5.4 .pt → ChaseTag ONNX (546 → 354)
import urllib.request
from pathlib import Path

from myosuite.integrations.musclemimic.fbvs_player import export_fbvs_onnx

ONNX_PATH = Path("chasetag_player.onnx")
# Your own 5.4 walker if you uploaded one, else the shipped one.
WALKERS = ("policy_bc_best.pt", "policy_heading_table.pt")
CKPT = next(
    (Path(d) / n for n in WALKERS for d in (".", "tutorials/files/5.6") if (Path(d) / n).is_file()),
    None,
)
if CKPT is None:
    CKPT = Path("policy_bc_best.pt")
    url = "https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/policy_bc_best.pt"
    print("Downloading the tutorial 5.4 walker …", url)
    urllib.request.urlretrieve(url, CKPT)
print("Walker:", CKPT.resolve())

# 546-d FBVs observation → 528-d walker observation (heading toward the opponent,
# away from it as runner) → walker → 354 muscle activations in [0, 1].
export_fbvs_onnx(CKPT, ONNX_PATH)
print(f"Wrote {ONNX_PATH.resolve()}  (obs [batch, 546] → action [batch, 354])")

In [ ]:
# @title Submit
import subprocess
import sys

cmd = [
    sys.executable, "-m", "evalmc", "submit-onnx", str(ONNX_PATH),
    "-t", "chasetag", "-p", "1",
    "-e", EMAIL, "-a", AUTH_TOKEN,
]
print("Uploading", ONNX_PATH.name, "…")
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise SystemExit(proc.returncode)
print("Upload accepted. Now open the website Submissions tab (next section).")


## 4. Watch evaluation on the website

Go back to [mc26.myosuite.org](https://mc26.myosuite.org) → **Submissions**. Refresh until the row moves **submitted → evaluating → scored**.

<p><img alt="Submissions table with status badges" width="440" src="https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/colab-watch.png"></p>

| Status | Meaning |
|---|---|
| **submitted** | queued |
| **evaluating** | running |
| **scored** | done — a number appears under Score |
| **failed** | click the row for the error |

This file is the 5.4 directional walker, not a random dummy. Train further in section 5, then re-run **Export** and **Submit**.


## 5. Train a stronger player

The upload above is the **tutorial 5.4 walker** with heading from opponent hip XY, flipped by the role flag (`[1, 0]` chase, `[0, 1]` evade).

To beat the baseline, train your own walker in 5.4, then re-run **Export** and **Submit**.

[![Open tutorial 5.4 in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.4_MuscleMimic_Directional_Locomotion.ipynb)

A stronger starting point: `chasetag_fbvs_baseline.onnx` is already fine-tuned on the scored body (`myofullbody`). Get it with `from myosuite.core.hf_io import download_baseline_file` and `ONNX_PATH = download_baseline_file("onnx/chasetag_fbvs_baseline.onnx")`, then re-run **Submit**. It tags a player that does nothing in about 3 s, but it often falls before the 10 s limit, so there is plenty to beat.

## 6. Play a local 1v1 and render a match (optional)

Plays your ONNX file as **both** players of `myoChallengeChaseTagFBVs-v0` (546-d) for a few episodes. Prints how each ended (tag, fall, time) with player A's score. Episode 0 is recorded until it ends.

Leave **Run** unchecked when you only want to submit (`Runtime → Run all`).


In [ ]:
# @title Play myoChallengeChaseTagFBVs-v0 locally and record a match
RUN = False  # @param {type:"boolean"}
ONNX_PATH = "chasetag_player.onnx"  # @param {type:"string"}
EPISODES = 4  # @param {type:"integer"}

if not RUN:
    print("Skipped. Check Run to play the 1v1 env locally and record a match.")
else:
    import os, sys, subprocess
    from pathlib import Path

    if sys.platform != "darwin":
        os.environ.setdefault("MUJOCO_GL", "egl")
        os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    if "google.colab" in sys.modules:
        subprocess.run(["apt-get", "-qq", "install", "-y", "libegl1", "libgles2"], check=False)
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "onnxruntime", "imageio", "imageio-ffmpeg",
        ])

    import imageio
    import mujoco
    import numpy as np
    import onnxruntime as ort
    from IPython.display import Video, display
    from myosuite import make_env

    onnx_path = Path(ONNX_PATH.strip())
    if not onnx_path.is_file():
        raise SystemExit(f"{onnx_path} not found: run the export cell or upload your .onnx first.")
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name

    def act(obs):
        x = np.asarray(obs, np.float32).reshape(1, -1)
        return np.clip(sess.run(None, {name: x})[0].reshape(-1), 0.0, 1.0)

    env = make_env("myoChallengeChaseTagFBVs-v0")
    raw = env.unwrapped

    renderer = mujoco.Renderer(raw.model, height=360, width=480)
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat[:] = [0.0, 0.0, 1.0]
    cam.distance = 5.0
    cam.azimuth = 135.0
    cam.elevation = -20.0

    frames = []
    print(f"{'ep':>3}  {'role':>5}  {'steps':>5}  {'t':>6}  {'end':<22}  {'score A':>7}")
    for episode in range(int(EPISODES)):
        obs, info = env.reset(seed=episode)
        n = 0
        while True:
            obs, _, terminated, truncated, info = env.step(
                {"agent_0": act(obs["agent_0"]), "agent_1": act(obs["agent_1"])}
            )
            n += 1
            if episode == 0:
                renderer.update_scene(raw.data, camera=cam)
                frames.append(np.asarray(renderer.render(), dtype=np.uint8))
            if terminated["agent_0"] or truncated["agent_0"]:
                break
        fell = [agent for agent, down in info["fell"].items() if down]
        end = "tag" if info["tagged"] else f"fall ({', '.join(fell)})" if fell else "time limit"
        print(f"{episode:3d}  {info['task']:5s}  {n:5d}  {info['elapsed_s']:5.2f}s  {end:<22}  {info['score']:7.3f}")
    renderer.close()
    env.close()

    out = Path("chasetag_fbvs.mp4")
    with imageio.get_writer(str(out), fps=30, quality=8, macro_block_size=1) as writer:
        for frame in frames[::3]:
            writer.append_data(frame)
    print("episode 0 recorded:", len(frames), "steps ->", out)
    display(Video(str(out), embed=True, html_attributes="controls loop"))
